In [1]:
from pymongo import MongoClient
import pandas as pd

# Connect to MongoDB and extract raw data
client = MongoClient('mongodb://localhost:27017/')
db = client['ecommerce_db']
collection = db['criteo_raw']

# Load all documents into a DataFrame
df = pd.DataFrame(list(collection.find()))

# Drop MongoDB's internal _id column
df = df.drop(columns=['_id'])

print("Extracted from MongoDB:")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nFirst 3 rows:")
print(df.head(3))

Extracted from MongoDB:
Shape: (100000, 22)

Columns: ['timestamp', 'uid', 'campaign', 'conversion', 'conversion_timestamp', 'conversion_id', 'attribution', 'click', 'click_pos', 'click_nb', 'cost', 'cpo', 'time_since_last_click', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cat9']

Data types:
timestamp                  int64
uid                        int64
campaign                   int64
conversion                 int64
conversion_timestamp       int64
conversion_id              int64
attribution                int64
click                      int64
click_pos                  int64
click_nb                   int64
cost                     float64
cpo                      float64
time_since_last_click      int64
cat1                       int64
cat2                       int64
cat3                       int64
cat4                       int64
cat5                       int64
cat6                       int64
cat7                       int64
cat8                    

In [2]:
#check for missing values
print(df.isnull().sum())

timestamp                0
uid                      0
campaign                 0
conversion               0
conversion_timestamp     0
conversion_id            0
attribution              0
click                    0
click_pos                0
click_nb                 0
cost                     0
cpo                      0
time_since_last_click    0
cat1                     0
cat2                     0
cat3                     0
cat4                     0
cat5                     0
cat6                     0
cat7                     0
cat8                     0
cat9                     0
dtype: int64


In [4]:
print("Missing values check:")
print(df.isnull().sum())
print("\nNo missing values found — dataset is clean, proceeding to type casting.")
print(f"\nActual columns in dataset: {df.columns.tolist()}")

Missing values check:
timestamp                    0
uid                          0
campaign                     0
conversion                   0
conversion_timestamp         0
conversion_id                0
attribution                  0
click                        0
click_pos                    0
click_nb                     0
cost                         0
cpo                          0
time_since_last_click    50177
cat1                         0
cat2                         0
cat3                         0
cat4                         0
cat5                         0
cat6                         0
cat7                         0
cat8                         0
cat9                         0
dtype: int64

No missing values found — dataset is clean, proceeding to type casting.

Actual columns in dataset: ['timestamp', 'uid', 'campaign', 'conversion', 'conversion_timestamp', 'conversion_id', 'attribution', 'click', 'click_pos', 'click_nb', 'cost', 'cpo', 'time_since_last_click', 'cat1

In [6]:
#added synthetic noise to data and performing summary stats 
# Extract from MongoDB

df = pd.DataFrame(list(collection.find()))
df = df.drop(columns=['_id'])

print("Noisy dataset extracted:")
print("Shape:", df.shape)
print("\nMissing values to clean:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nDuplicate rows to remove: {df.duplicated().sum():,}")
print(f"Sentinel -1 values in time_since_last_click: {(df['time_since_last_click'] == -1).sum():,}")

Noisy dataset extracted:
Shape: (102000, 22)

Missing values to clean:
campaign    3000
cost        5033
cpo         4081
dtype: int64

Duplicate rows to remove: 1,857
Sentinel -1 values in time_since_last_click: 55,250


In [7]:
#perform data cleaning

# ── Clean 1: Remove duplicates ───────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {before - len(df):,} rows")

# ── Clean 2: Replace sentinel -1 with None ──────────────────────────────────
sentinel_count = (df['time_since_last_click'] == -1).sum()
df['time_since_last_click'] = df['time_since_last_click'].replace(-1, None)
print(f"Sentinel values replaced: {sentinel_count:,}")

# ── Clean 3: Handle missing cost values — fill with median ──────────────────
cost_median = df['cost'].median()
missing_cost = df['cost'].isnull().sum()
df['cost'] = df['cost'].fillna(cost_median)
print(f"Missing cost filled with median ({cost_median:.4f}): {missing_cost:,} rows")

# ── Clean 4: Handle missing cpo values — fill with median ───────────────────
cpo_median = df['cpo'].median()
missing_cpo = df['cpo'].isnull().sum()
df['cpo'] = df['cpo'].fillna(cpo_median)
print(f"Missing cpo filled with median ({cpo_median:.4f}): {missing_cpo:,} rows")

# ── Clean 5: Fix inconsistent campaign formatting ────────────────────────────
before_camp = df['campaign'].isnull().sum()
df['campaign'] = df['campaign'].astype(str).str.replace('CAMP_', '', regex=False).str.upper()
df['campaign'] = df['campaign'].replace('NAN', None)
df['campaign'] = df['campaign'].fillna('UNKNOWN')
print(f"Campaign formatting standardised, missing filled: {before_camp:,} rows")

# ── Clean 6: Handle invalid timestamps ──────────────────────────────────────
invalid_ts = (df['timestamp'] == -1).sum()
df['timestamp'] = df['timestamp'].replace(-1, None)
print(f"Invalid timestamps set to null: {invalid_ts:,}")

# ── Clean 7: Remove cost outliers (above 99th percentile) ───────────────────
cost_99 = df['cost'].quantile(0.99)
outlier_count = (df['cost'] > cost_99).sum()
df = df[df['cost'] <= cost_99]
print(f"Outliers removed (cost > {cost_99:.4f}): {outlier_count:,} rows")

print(f"\nCleaning complete.")
print(f"Final shape: {df.shape}")
print(f"Remaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Duplicates removed: 1,857 rows
Sentinel values replaced: 54,261
Missing cost filled with median (0.0001): 4,946 rows
Missing cpo filled with median (0.1669): 4,003 rows
Campaign formatting standardised, missing filled: 3,000 rows
Invalid timestamps set to null: 2,000
Outliers removed (cost > 0.0281): 1,002 rows

Cleaning complete.
Final shape: (99141, 22)
Remaining missing values:
timestamp                 1982
time_since_last_click    53715
dtype: int64
